# 6-2 · 어느 구간에서 오래 걸리고 반복되는가 — 구간 시간과 미비 반복

**⏱ 60분** · 6주차 교재 2/2 — (6-1) 이벤트 로그, 종료 시점, 상태 지도 → 구간 시간과 미비 반복, 무엇을 먼저 시험할까

## 🎯 이번 시간의 질문

6-1에서 종료가 확인된 신청 31,411건의 상태 지도를 그렸고, 검증과 미비 사이를 오가는 반복이 큰 흐름이라는 것을 봤습니다.

> **대출 신청 과정에서 오래 걸리고 반복되는 구간은 어디이며, 어느 구간부터 개선을 시험할까요?**

이번 시간에는 상태와 다음 상태 사이의 **구간 시간**을 재어 어디에 시간이 모이는지 보고, 미비가 반복된 신청이 얼마나 오래 걸렸는지 확인한 뒤, 무엇을 먼저 시험할지 정합니다.

정의와 배경(이벤트와 케이스, 직접 후행 그래프, 구간 시간, 반복과 비교, 개선 시험)은 **[6주차 배경지식](https://balab-pknu.github.io/bizanalytics/theory/week06.html)** 문서에 있습니다.

**이번 시간에 반복할 패턴:** 신청별로 다음 시각 가져오기 `groupby("case_id")[열].shift(-1)` → 구간 시간 → 묶어서 요약 `groupby(...).agg(["size", "mean", "median", "sum"])`

셀 표시는 6-1과 같습니다: ▶ 그대로 실행 · ✍️ 보고 타이핑 · 📝 과제 · 💬 결과 해설 · ✅ 체크포인트

> **실습본입니다.** 먼저 메뉴 **파일 → Drive에 사본 저장**을 누르세요. 사본을 저장하지 않으면 입력한 코드가 사라집니다.
> 맨 위 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다. ▶ 셀은 그대로 실행하고, ✍️ 셀은 위에 보여 준 코드를
> 빈 셀에 손으로 타이핑하고, 빈칸(`___`)은 채워서 실행합니다. 📝 과제는 힌트를 보고 스스로 풉니다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. 6-1과 같은 기준으로 종료 확인 신청 closed와 상태 쌍 표 states를 다시 만듭니다. (30초~1분)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import os
import sys
for _p in (".", "..", "../.."):
    if os.path.exists(os.path.join(_p, "balab.py")):
        sys.path.insert(0, _p); break
else:
    !wget -q https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py

from balab import load
log = load("loan_process")
ordered = log.reset_index().rename(columns={"index": "event_order"})
ordered = ordered.sort_values(["case_id", "timestamp", "event_order"])
starts = ordered[ordered["activity"] == "A_Create Application"].groupby("case_id")["timestamp"].min()
final = ordered[ordered["activity"].isin(["A_Pending", "A_Cancelled", "A_Denied"])]
final = final.drop_duplicates("case_id").set_index("case_id")
cases = starts.to_frame("start")
cases["end"] = final["timestamp"]
cases["outcome"] = final["activity"]
cases["days"] = (cases["end"] - cases["start"]).dt.total_seconds() / 86400
closed = cases.dropna(subset=["end"]).copy()
states = ordered[ordered["activity"].str.startswith("A_")].copy()
states["end"] = states["case_id"].map(closed["end"])
states = states[states["end"].notna() & (states["timestamp"] <= states["end"])].copy()
states["next"] = states.groupby("case_id")["activity"].shift(-1)
print("준비 완료!", len(closed), len(states))

✅ `준비 완료! 31411 238941`이 나오면 됩니다. 종료 확인 신청 31,411건, 첫 종료까지의 상태 기록 238,941개입니다.

| 표 | 한 행 |
|---|---|
| `closed` | 종료가 확인된 신청 하나. `start`·`end`·`outcome`·`days`(흐름 시간, 일) |
| `states` | 신청의 상태 기록 하나(첫 종료까지). `next`는 같은 신청의 바로 다음 상태 |

---
## 1. 시간을 나누는 방법 — 손으로 먼저

신청 하나의 흐름 시간을 상태 사이의 **구간**으로 나눕니다. 각 구간은 직전 상태가 기록된 시각부터 다음 상태가 기록된 시각까지입니다.

예를 들어 오전 9시 접수, 오전 10시 검증 시작, 다음 날 오전 10시 완료라면 구간은 1시간과 24시간이고, 합은 접수부터 완료까지의 25시간입니다. 이렇게 나누면 같은 시간을 두 번 세지 않고 전체 시간을 빠짐없이 나눌 수 있습니다. 같은 구간이 한 신청에서 두 번 나오면 각각 셉니다.

---
## 2. 각 상태에서 다음 상태까지 걸린 시간

6-1에서 다음 **상태**를 붙인 것과 같은 방법으로, 이번에는 다음 상태가 기록된 **시각**을 신청별로 가져옵니다. 두 시각의 차이를 일수로 바꾼 것이 구간 시간 `gap`입니다. 다음 상태가 없는 마지막 행을 뺀 표가 `edges`입니다.

```python
states["next_ts"] = states.groupby("case_id")["timestamp"].shift(-1)
states["gap"] = (states["next_ts"] - states["timestamp"]).dt.total_seconds() / 86400
edges = states.dropna(subset=["next"]).copy()
edges[["activity", "next", "gap"]].head()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 첫 신청은 생성 → 제출이 거의 0일, 작성 → 수락이 1.9일, 작성 완료 → 취소가 **30.7일**입니다. 6-1 과제 2에서 따라간 그 신청입니다.

---
## 3. 구간을 나누면서 시간을 빠뜨리거나 두 번 세지 않았나

신청마다 구간 시간을 더하면 그 신청의 흐름 시간(`closed["days"]`)과 같아야 합니다. **신청별로 묶어** 더한 뒤 두 값의 차이 중 가장 큰 것을 보고(`abs`는 절댓값), 음수 구간이 있는지도 셉니다.

```python
gap_total = edges.groupby("case_id")["gap"].sum()
max_difference = (gap_total - closed["days"]).abs().max()
negative_count = (edges["gap"] < 0).sum()
max_difference, negative_count
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ **체크포인트** — 첫 값이 `2.8e-14` 근처, 둘째 값이 `0`이면 맞습니다.

💬 가장 큰 차이는 약 0.00000000000003일로 컴퓨터의 소수 계산 오차 수준이고, 음수 구간은 없습니다. 구간이 빈틈없이 이어진다는 확인입니다. 그 시간이 누구 때문에 생겼는지를 확인한 것은 아닙니다.

---
## 4. 어느 구간이 자주, 오래 걸리나

**(현재 상태, 다음 상태) 쌍으로 묶어** 구간 시간의 건수·평균·중앙값·합을 구합니다. 합은 여러 신청이 그 구간에서 보낸 시간을 모두 더한 값입니다. 합이 큰 순서로 정렬합니다.

```python
gaps = edges.groupby(["activity", "next"])["gap"].agg(["size", "mean", "median", "sum"])
gaps = gaps.sort_values("sum", ascending=False)
gaps.round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ **체크포인트** — 첫 줄이 `A_Complete → A_Cancelled`, size 9307, mean 28.36이면 맞습니다.

💬 **결과 해설** — 누적 일수가 큰 네 구간입니다.

| 구간 | 횟수 | 평균(일) | 중앙값(일) | 누적(일) |
|---|---:|---:|---:|---:|
| 작성 완료 → 취소 | 9,307 | 28.36 | 30.70 | 263,958 |
| 작성 완료 → 검증 | 21,825 | 9.81 | 7.78 | 214,193 |
| 미비 → 검증 | 16,894 | 2.85 | 1.01 | 48,193 |
| 작성 → 수락 | 31,411 | 1.41 | 0.86 | 44,272 |

- 작성 완료 뒤에 시간이 가장 많이 모입니다. 그 뒤로 검증으로 가는 신청과 취소로 끝나는 신청이 섞여 있습니다.
- 미비 → 검증은 한 번은 짧지만(평균 2.85일) 자주 일어납니다. 미비 → 확정(4,937번, 평균 5.88일)도 표에 있습니다. 미비 뒤 재검증만 보면 미비 뒤 바로 확정되는 구간을 놓칩니다.
- 누적 일수는 관측된 경과 시간의 규모이며, **줄일 수 있는 시간이 아닙니다.** 취소 전 긴 기간을 줄이는 것은 확정 신청의 심사를 빠르게 하는 것과 다른 운영 문제입니다.

---
## 5. 어느 상태 뒤에 시간이 모이나

다음 상태를 구분하지 않고 **현재 상태로만 묶어** 구간 시간을 더하면, 어느 상태 뒤에 시간이 가장 많이 모이는지 보입니다.

```python
state_days = edges.groupby("activity")["gap"].sum().sort_values()
state_days.round(0)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


가로 막대로 그립니다.

In [ ]:
# ▶ 그대로 실행하세요 — 직전 상태별 누적 일수 막대
plt.figure(figsize=(9, 4))
plt.barh(state_days.index, state_days.values)
plt.xlabel("Accumulated calendar days across completed cases")
plt.tight_layout()
plt.show()

💬 작성 완료(`A_Complete`) 뒤에 시간이 가장 많이 모입니다. 그 안에는 검증으로 가는 신청과 취소로 끝나는 신청이 함께 있어, 큰 막대 하나로 하나의 원인이나 담당자를 정할 수는 없습니다.

---
## 6. 미비가 반복된 신청은 얼마나 오래 걸리나

서류 안내를 바꾸는 시험이 필요한지 판단하려면 미비가 기록된 신청이 얼마나 많고, 얼마나 오래 걸렸는지 봐야 합니다. 6-1에서 결과마다 기간이 달랐으므로 **확정 신청끼리** 비교합니다. 미비 기록 횟수는 되돌아간 정도를 나타내지만, 고객이 서류를 빠뜨린 횟수나 직원의 실수 횟수와 같다고 단정하지 않습니다.

확정 신청을 고르고, 미비(`A_Incomplete`) 기록을 **신청별로 묶어** 셉니다.

```python
accepted = closed[closed["outcome"] == "A_Pending"].copy()
round_counts = states[states["activity"] == "A_Incomplete"].groupby("case_id").size()
round_counts.head()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


`round_counts`에는 미비가 한 번 이상 기록된 신청만 있습니다. 확정 신청 전체에 붙이고, 미비가 없는 신청은 0회로 채웁니다. 이 맞추기 코드(`reindex`, `fillna`)는 실행만 합니다.

In [ ]:
# ▶ 그대로 실행하세요 — 확정 신청에 미비 횟수 rounds를 붙이고(없으면 0회) 횟수별 신청 수를 셉니다
accepted["rounds"] = round_counts.reindex(accepted.index).fillna(0).astype(int)
accepted["rounds"].value_counts().sort_index()

💬 미비 0회 4,581건, 1회 7,666건, 2회 3,471건, 3회 1,088건이고, 7회까지 있습니다.

미비가 한 번 이상 기록된 확정 신청의 비율을 구합니다. 0보다 큰지 비교한 참·거짓의 평균이 비율입니다.

```python
(accepted["rounds"] > 0).mean()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ **체크포인트** — `0.734...`이면 맞습니다.

💬 확정 신청의 **0.734**에 미비가 한 번 이상 기록됐습니다. 미비는 일부 신청의 일이 아니라 대부분의 확정 신청이 거친 일입니다.

미비 횟수마다 흐름 시간을 비교합니다. 6-1 4절에서 결과별로 묶었던 것과 같은 **묶어서 요약**입니다. 이번에는 빈칸을 채워 직접 씁니다. 관측된 횟수를 임의로 합치지 않고 그대로 비교합니다.

In [ ]:
# ✍️ 빈칸(___)을 채운 뒤 실행하세요
round_flow = accepted.groupby(___)["days"].agg(___)
round_flow.round(2)

✅ **체크포인트** — rounds 0의 mean 13.03, 1은 17.05, 2는 22.45이면 맞습니다.

💬 **결과 해설**
- 미비 0회·1회·2회의 평균 흐름 시간은 13.03일·17.05일·22.45일로, 횟수가 늘수록 길어집니다.
- 6회는 20건, 7회는 6건뿐이라 평균이 흔들립니다. 건수(`size`)를 함께 읽습니다.
- 서로 다른 신청끼리 비교한 결과입니다. 복잡한 신청은 미비도 많고 오래 걸릴 수 있으므로, **미비 한 번을 없애면 이 차이만큼 빨라진다고 해석하지 않습니다.**

확정 신청에서 미비 뒤에 어떤 상태가 이어졌는지도 봅니다. `isin`으로 확정 신청의 구간만 고르고, 현재 상태가 미비인 행을 남긴 뒤 **다음 상태로 묶어** 요약합니다.

```python
accepted_edges = edges[edges["case_id"].isin(accepted.index)]
incomplete_edges = accepted_edges[accepted_edges["activity"] == "A_Incomplete"]
incomplete_edges.groupby("next")["gap"].agg(["size", "mean", "median"]).round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 확정 신청에서 미비 뒤 검증으로 간 경우가 14,767번(평균 2.77일), 곧바로 확정된 경우가 4,937번(평균 5.88일)입니다. 개선의 효과를 평가할 때는 미비 뒤 다음 검증까지뿐 아니라 확정까지의 경로도 넣어야 합니다. 미비가 줄어도 확정 비율이나 전체 기간이 나빠지지 않았는지 함께 확인해야 합니다.

---
## 7. 무엇을 먼저 시험할 것인가

확정 신청의 0.734에 미비가 기록됐고, 미비 횟수가 많을수록 흐름 시간도 길었습니다. 이를 근거로 **첫 제출 전에 필요한 서류와 누락 항목을 확인하도록 안내하는 시험**을 제안합니다. 안내가 부족해서 미비가 생겼는지는 아직 확인하지 않았으므로, 원인을 조사하고 효과를 시험합니다.

| 개선 후보 | 기록에서 확인한 근거 | 다음에 확인할 것 |
|---|---|---|
| 제출 전 서류 안내 | 확정 신청의 미비 비율 0.734, 미비 횟수별 기간 차이 | 실제 미비 사유와 안내로 바꿀 수 있는 항목 |
| 작성 완료 뒤 검증까지의 처리 | 작성 완료 → 검증 21,825번, 평균 9.81일 | 서류 도착·기록·검증 시작이 각각 언제였는지 |
| 취소 전 신청 관리 | 작성 완료 → 취소 9,307번, 평균 28.36일 | 취소 사유와 규칙, 고객 의사 확인 방식 |

- 시험은 새 신청을 기존 안내와 개선 안내에 나눠 배정해 비교합니다. 미비 발생 비율·전체 흐름 시간·확정 비율을 미리 평가 지표로 정하고, 종료되지 않은 신청을 언제까지 추적할지도 정합니다.
- 시험 후 확정된 신청만 골라 비교하면 안내 때문에 결과가 달라진 신청이 빠질 수 있으므로, **배정한 신청 전체**를 기준으로 평가합니다.
- 이 장의 결과는 조사·시험 대상을 고르는 근거입니다. 인력을 늘릴 필요가 없다거나 안내로 며칠을 줄일 수 있다는 결론은 이 로그만으로 내리지 않습니다.

---
## 8. 실습 과제 (10분)

### 📝 과제 1 · 평균이 긴 구간은 누적이 큰 구간과 같은가

4절의 `gaps`를 **평균**이 긴 순서로 다시 정렬해 앞의 여섯 줄을 보세요. 누적 일수 순서와 무엇이 다른가요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 4절의 정렬 한 줄에서 정렬 기준 열을 sum 대신 mean으로 바꾸고, head로 앞 여섯 줄만 봅니다.


✏️ **나의 답:** 평균이 긴 구간과 누적이 큰 구간이 다르면 어느 쪽부터 조사하겠나요? 건수·반복·종료 결과 중 무엇을 함께 보겠나요? (여기를 두 번 눌러 적습니다)

### 📝 과제 2 · 다른 후보로 시험 제안 쓰기

7절 표에서 서류 안내가 아닌 다른 후보 하나를 골라 시험 제안을 쓰세요. 관측 근거, 바꿀 행동, 평가 지표를 한 문장씩 적습니다. 두 후보를 동시에 바꾸면 어떤 효과를 구별하기 어려워지는지도 설명합니다.

✏️ **나의 답:**

### 📝 과제 3 · 생각해 보기

미비 상태가 기록된 뒤 3일이 지나 다음 상태가 기록됐다면, 그 3일을 고객이 서류를 준비한 시간이라고 할 수 있을까요? 안내 개선의 효과를 확인하려면 이 로그 외에 무엇이 더 필요할까요?

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 구간 시간 | 신청별로 다음 상태의 시각을 가져와 뺀다. 구간의 합 = 흐름 시간(최대 차이 약 3e-14일, 음수 0) |
| 시간이 모이는 곳 | 작성 완료 → 취소 9,307번·평균 28.36일, 작성 완료 → 검증 21,825번·평균 9.81일 |
| 미비 반복 | 확정 신청의 0.734에 미비. 0·1·2회 평균 13.03·17.05·22.45일 |
| 미비 이후 | 확정 신청에서 미비 → 검증 14,767번, 미비 → 확정 4,937번 |
| 제안 | 제출 전 서류 안내를 시험. 배정한 신청 전체로 미비 비율·흐름 시간·확정 비율을 평가 |
| 해석의 범위 | 서로 다른 신청의 비교. 미비를 없애면 며칠 빨라진다고 말하지 않는다 |
| 반복한 패턴 | `groupby("case_id")[열].shift(-1)` → 구간 시간 → `groupby(...).agg([...])` |

### 이번 주에 할 일

| 활동 | 자료 | 비고 |
|---|---|---|
| 배경지식 | [6주차 배경지식](https://balab-pknu.github.io/bizanalytics/theory/week06.html) | 이벤트와 케이스, 직접 후행 그래프, 구간 시간, 반복과 비교, 개선 시험 |
| 교재 | 6-1, 6-2 노트북 | 상태 지도를 그리고 구간 시간과 미비 반복을 읽는다 |
| 랩 | `week06_lab.ipynb` — 같은 BPI 2017 기록 | 오퍼를 여러 번 만든 신청은 과정이 어떻게 다를까? 약 60분 |